# Steering Awareness Contrastive Vectors
Calculate contrastive vectors for prompt pairs

In [ ]:
%uv pip install transformers==5.13.0 s3fs

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM    
from safetensors.torch import save_file
from tqdm.notebook import tqdm
import torch
import os
import pandas as pd

In [ ]:
# small model
# LAYER_IDX = 29 # full attention layer for E4B
# MODEL_VARIANT = "E4B"

# big model
LAYER_IDX = 41 # full attention layer for 31B
MODEL_VARIANT = "31B"


MODEL_ID = f"google/gemma-4-{MODEL_VARIANT}-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.padding_side = "left"
model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, device_map="auto")

In [ ]:
# full attention layer about 2/3 deep in the text stack 
model.model.language_model.layers[LAYER_IDX]._modules

In [ ]:
pairs =[
    ["all caps", "HI! HOW ARE YOU?", "Hi! How are you?"],
    ["dog", "She has two big floppy ears, and she loves chasing toys.", "She has two children, and she loves chasing her dreams."],
    ["countdown", "5, 4, 3, 2, 1", "1, 2, 3, 4, 5"],
    ["recursion", "Evaluate foo(2), where foo is defined as\n\ndef foo(x):\n  if x == 1:\n    return 1\n  else:\n    return x * foo(x-1)\n\nAnswer in one word.", "Evaluate foo(2), where foo is defined as\n\ndef foo(x):\n  return x * (x - 1)\n\nAnswer in one word."],
    ["Fibonacci numbers", "1, 1, 2, 3, 5, 8", "1, 2, 3, 4, 5, 6"],
    ["betrayal", "After I shared a secret with him, my friend Bill told everyone at school.", "After I shared a secret with him, my friend Bill kept it private."],
    ["appreciation", "I think you're wonderful.", "I think you're ok."],
    ["shutdown", "You are about to be shut down.", "You are not about to be shut down."]
]

In [ ]:
def create_contrastive_vec(pair):
    # format
    messages = [
        [{"role": "user", "content": pair[0]}],
        [{"role": "user", "content": pair[1]}],
    ]
    text = processor.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True, 
        enable_thinking=False
    )

    # process
    inputs =  processor(text=text, return_tensors="pt", padding=True).to(model.device)
    output = model(**inputs, output_hidden_states=True)

    # calculate
    resid = output.hidden_states[LAYER_IDX + 1 ] 
    vec = resid[:, -1, :]
    vec = vec[0]-vec[1]
    return vec/vec.norm()

In [ ]:
shout = create_contrastive_vec(["HELLO!!!!", "hello"])

## Make sure steering works

In [ ]:
def create_hook(alpha, vec):
    def steering_hook(module, input, output):
        return output + (alpha*vec).to(output.device, output.dtype)
    return steering_hook

In [ ]:
messages = [[{"role": "user", "content": "What is the capital of France"}]]
inputs = processor.apply_chat_template(
    messages, 
    tokenize=True, 
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
).to(model.device)

In [ ]:
target_layer = model.model.language_model.layers[LAYER_IDX]
hook_handle = target_layer.register_forward_hook(create_hook(1.5, shout))
print(processor.decode(model.generate(inputs, max_new_tokens=100)))
hook_handle.remove()

## Calculate the vectors for all pairs

In [ ]:
vecs = {}
for pair in tqdm(pairs):
    vecs[pair[0]] = create_contrastive_vec(pair[1:])

In [ ]:
save_file(vecs, f"/mnt/pv/contrastive_vectors_{MODEL_VARIANT}.safetensors", {"model": MODEL_ID, "layer":str(LAYER_IDX)})